In [41]:
import time
import pyautogui as pyg

# =========================================================
# 1. 座標與區域截圖設定
# =========================================================
baseX = 518      # 第一顆寶石 (row 0, col 0) 的 X 座標
baseY = 384      # 第一顆寶石 (row 0, col 0) 的 Y 座標
gap = 40         # 寶石中心間距 (像素)
GAME_TIME = 65   # 遊戲時間 (秒)[cite: 6]

pyg.PAUSE = 0.0001  # 指令發射零延遲

# 設定 8x8 棋盤的極速區域截圖範圍 (縮小截圖範圍以提升 3 倍截圖速度)
REGION_LEFT = baseX - 15
REGION_TOP = baseY - 15
REGION_WIDTH = 8 * gap + 30
REGION_HEIGHT = 8 * gap + 30

banned_moves = {}

def get_gem_position(r, c):
    """計算螢幕絕對 X, Y 座標"""
    return (baseX + c * gap, baseY + r * gap)

def get_gem_color_fast(r, c, im):
    """從區域截圖中快速取得相對 RGB 顏色"""
    # 換算為區域截圖內部的相對座標
    rel_x = c * gap + 15
    rel_y = r * gap + 15
    return im.getpixel((rel_x, rel_y))

# =========================================================
# 2. 靜止盤面與色彩判斷
# =========================================================
def is_background(color):
    if color is None:
        return True
    return (color[0] + color[1] + color[2]) < 85

def colors_match(c1, c2, tolerance=18):
    if is_background(c1) or is_background(c2):
        return False
    return all(abs(a - b) <= tolerance for a, b in zip(c1[:3], c2[:3]))

def get_stable_board_fast():
    """極速區域雙重截圖防抖"""
    im1 = pyg.screenshot(region=(REGION_LEFT, REGION_TOP, REGION_WIDTH, REGION_HEIGHT))
    board1 = [[get_gem_color_fast(r, c, im1) for c in range(8)] for r in range(8)]
    
    pyg.sleep(0.015)  # 縮短至 15ms 驗證
    
    im2 = pyg.screenshot(region=(REGION_LEFT, REGION_TOP, REGION_WIDTH, REGION_HEIGHT))
    board2 = [[get_gem_color_fast(r, c, im2) for c in range(8)] for r in range(8)]
    
    diff_count = 0
    for r in range(8):
        for c in range(8):
            if not colors_match(board1[r][c], board2[r][c], tolerance=10):
                diff_count += 1
                if diff_count > 2:
                    return None
                
    return board2

# =========================================================
# 3. 權重評估矩陣 (包含 Multiplier 與塌陷優先)
# =========================================================
def check_match_at(r, c, board):
    color = board[r][c]
    if is_background(color):
        return 0

    h_count = 1
    nc = c - 1
    while nc >= 0 and colors_match(board[r][nc], color):
        h_count += 1
        nc -= 1
    nc = c + 1
    while nc < 8 and colors_match(board[r][nc], color):
        h_count += 1
        nc += 1

    v_count = 1
    nr = r - 1
    while nr >= 0 and colors_match(board[nr][c], color):
        v_count += 1
        nr -= 1
    nr = r + 1
    while nr < 8 and colors_match(board[nr][c], color):
        v_count += 1
        nr += 1

    if h_count >= 5 or v_count >= 5: return 10000  # Hypercube[cite: 4]
    if h_count >= 3 and v_count >= 3: return 7000   # Star Gem[cite: 4]
    if h_count >= 4 or v_count >= 4: return 4000   # Flame Gem[cite: 3]
    if h_count >= 3 or v_count >= 3: return 1000   # 3 Match[cite: 3]

    return 0

def evaluate_move(r, c, dr, dc, board, current_time):
    nr, nc = r + dr, c + dc
    if not (0 <= nr < 8 and 0 <= nc < 8):
        return 0

    m_key1 = (r, c, nr, nc)
    m_key2 = (nr, nc, r, c)
    if m_key1 in banned_moves and current_time < banned_moves[m_key1]:
        return 0
    if m_key2 in banned_moves and current_time < banned_moves[m_key2]:
        return 0

    c1 = board[r][c]
    c2 = board[nr][nc]

    if is_background(c1) or is_background(c2):
        return 0
    if colors_match(c1, c2):
        return 0

    sim_board = [list(row) for row in board]
    sim_board[r][c], sim_board[nr][nc] = sim_board[nr][nc], sim_board[r][c]

    score1 = check_match_at(r, c, sim_board)
    score2 = check_match_at(nr, nc, sim_board)

    base_score = max(score1, score2)
    if base_score == 0:
        return 0

    # 底部塌陷大加權 (引發連鎖與掉落 Multiplier)[cite: 4]
    bottom_weight = (max(r, nr) + 1) * 80
    return base_score + bottom_weight

# =========================================================
# 4. 極速主程式 (區域截圖 + 多重預連擊)
# =========================================================
def main():
    print(f"基準座標: baseX={baseX}, baseY={baseY}")
    print("程式將在 3 秒後開始執行，請切換至 Bejeweled Blitz 視窗...")
    pyg.sleep(3)
    print("【啟動極速區域截圖 + 多重預連擊飆分模式！】")

    start_time = time.time()
    directions = ((0, 1), (1, 0), (0, -1), (-1, 0))

    while time.time() - start_time < GAME_TIME:
        now = time.time()
        
        # 1. 極速區域截圖與防抖
        board = get_stable_board_fast()
        if board is None:
            pyg.sleep(0.008)  # 8ms 極短微調
            continue

        possible_moves = []
        for r in range(8):
            for c in range(8):
                for dr, dc in directions:
                    weight = evaluate_move(r, c, dr, dc, board, now)
                    if weight > 0:
                        possible_moves.append((weight, r, c, dr, dc))

        if possible_moves:
            possible_moves.sort(key=lambda x: x[0], reverse=True)
            
            # 2. 多重預連擊機制：一次挑選最多 2 組不重疊步法連續發射
            used_coords = set()
            click_queue = []
            
            for w, r, c, dr, dc in possible_moves:
                nr, nc = r + dr, c + dc
                if (r, c) not in used_coords and (nr, nc) not in used_coords:
                    click_queue.append((r, c, nr, nc))
                    used_coords.add((r, c))
                    used_coords.add((nr, nc))
                    if len(click_queue) >= 2:  # 最多連發 2 組
                        break

            # 3. 發射點擊
            for r, c, nr, nc in click_queue:
                banned_moves[(r, c, nr, nc)] = now + 2.5
                x1, y1 = get_gem_position(r, c)
                x2, y2 = get_gem_position(nr, nc)
                pyg.click(x1, y1)
                pyg.click(x2, y2)

            # 極短緩衝，持續拉滿 Blazing Speed[cite: 5]
            pyg.sleep(0.05)
        else:
            pyg.sleep(0.01)

    print("【遊戲時間結束！】")

if __name__ == "__main__":
    main()

基準座標: baseX=518, baseY=384
程式將在 3 秒後開始執行，請切換至 Bejeweled Blitz 視窗...
【啟動極速區域截圖 + 多重預連擊飆分模式！】


KeyboardInterrupt: 